Introduction
============
**Topic**: Setting up a simple 2D environment for testing collisions.

This notebook is part of the interactive worksheets regarding "Introduction to path planning".

Version | Author
------------ | -------------
0.3 | Björn Hein

One goal of the notebook series is to discuss and implement algorithms for collision-free path planning. Therefore a concept is needed to represent obstacles and check whether sampled configurations or local connections are valid. This notebook gives a simple implementation of such an environment using the Python library **Shapely**.

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (pls. check: http://creativecommons.org/licenses/by-nc/4.0/)

Important links are:

* General Info: https://www.python.org
* NetworkX: https://networkx.org
* NumPy: https://numpy.org/doc/stable/
* Matplotlib: https://matplotlib.org/stable/
* Shapely: https://shapely.readthedocs.io/

# Goal

* Understanding how to set up a simple 2D testing environment
* Understanding how to visualize the environment
* Understanding how to test points and local connections for collision, and why the number of collision checks matters
* Understanding how to wrap these checks in a collision checker for path planning algorithms
* Understanding how to define reusable benchmark scenes with valid start and goal configurations, and what makes a benchmark difficult

Set up the Environment
======================

The planning will take place in a certain environment. For testing it is sometimes useful to simplify the environment, e.g. taking a simple 2D approach: the robot is a point, so the work space is identical to the configuration space (see IP-2-0).

**Remember**: Even if the obstacles are drawn in this notebook, the investigated algorithms must behave as if these **obstacles are not explicitly available**. A planner should only ask whether a configuration or a local connection is valid. This corresponds to an implicit configuration space.

The 2D Python library Shapely is a good choice for this purpose (https://shapely.readthedocs.io/).

Shapely is a BSD-licensed Python package for manipulation and analysis of planar geometric objects. It is based on the widely deployed GEOS library.

Shapely provides much more functionality than needed here. In the context of this lecture, it will mainly be used to test geometric objects for collision.

Idea
-----

The idea is to use the functionality provided by Shapely and define the environment programmatically. Obstacles are stored in a dictionary so that scenes can be passed to a planner or benchmark later.

In [ ]:
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, LineString, box
from shapely.ops import unary_union
from shapely import plotting

# colors used in all notebooks
OBSTACLE_COLOR = "#52514e"
START_COLOR = "#0ca30c"
GOAL_COLOR = "#d03b3b"
FREE_COLOR = "#2a78d6"        # configuration checked: free
COLLISION_COLOR = "#eb6834"   # configuration checked: in collision

LIMITS = [[0.0, 22.0], [0.0, 22.0]]   # limits of the environment (same as in IPEnvironment.py)

Defining a scene
-----------------

1. Definition of obstacles via polygons.
2. As storage a dict is proposed containing then the "scene"

In [ ]:
simpleField = dict()
simpleField["obs1"] = Polygon([(6, 14), (13, 14), (13, 11), (6, 11)]).buffer(.02)
simpleField["obs2"] = Polygon([(16, 5), (19, 5), (19, 3), (16, 3)]).buffer(1.0)

Based on that scene definition, a simple visualization function can be written. It is used for all drawings in this notebook.

In [ ]:
def drawScene(scene, limits=LIMITS, figsize=(5, 5), starts=None, goals=None, title=None, ax=None):
    """Draw the obstacles of a scene (and optionally start and goal configurations). Returns the axes."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(limits[0])
    ax.set_ylim(limits[1])
    ax.set_aspect("equal")
    ax.grid(True, linewidth=0.3)
    for key, value in scene.items():
        plotting.plot_polygon(value, add_points=False, ax=ax, color=OBSTACLE_COLOR)
    if starts:
        ax.scatter([p[0] for p in starts], [p[1] for p in starts], color=START_COLOR, s=80, zorder=5, label="start")
    if goals:
        ax.scatter([p[0] for p in goals], [p[1] for p in goals], color=GOAL_COLOR, s=80, zorder=5, label="goal")
    if starts or goals:
        ax.legend()
    if title:
        ax.set_title(title)
    return ax

In [ ]:
ax = drawScene(simpleField, figsize=(3, 3))

A scene with a trap: a planner that heads straight from the start (inside) towards a goal below runs into the "U". This scene is used later in the benchmark "Trap".

In [ ]:
trapField = dict()
trapField["obs1"] = LineString([(6, 18), (6, 8), (16, 8), (16, 18)]).buffer(1.0)

ax = drawScene(trapField, figsize=(4, 4))

For a more complicated example see the following "Christmas tree": many dead ends (the pockets between the branches) and very thin obstacles (the "candles", only 0.1 wide). Both will be used below.

In [ ]:
treeField = dict()
treeField["trunk"] = LineString([(11, 0), (11, 18)]).buffer(0.5)
for i in range(15):
    target1 = (11 - 0.5 * i, 18 - i)
    target2 = (11 + 0.5 * i, 18 - i)
    treeField["branchL" + str(i)] = LineString([(11, 18 - i), target1]).buffer(0.2)
    treeField["branchR" + str(i)] = LineString([(11, 18 - i), target2]).buffer(0.2)
    if i % 2 == 0:
        treeField["candleL" + str(i)] = LineString([target1, (target1[0], target1[1] + 1.5)]).buffer(0.05)
        treeField["candleR" + str(i)] = LineString([target2, (target2[0], target2[1] + 1.5)]).buffer(0.05)

ax = drawScene(treeField, figsize=(7, 7))

Collision checking
=================

One advantage of using shapely is the ability to easily check geometric objects for **intersection** and **distance**.

Point - Point
---------------

A shapely `Point` has no extent. `buffer(r)` turns it into a disc with radius `r`.

In [ ]:
object1 = Point([13, 5]).buffer(0.01)
object2 = Point([10, 5]).buffer(4.)

print("intersects:", object1.intersects(object2))
print("distance:", object1.distance(object2))

ax = drawScene({"ob1": object1, "ob2": object2}, figsize=(3, 3))

Disc - Line example
-----------------------

In [ ]:
lineScene = dict()
lineScene["disc"] = Point([10, 5]).buffer(2.0)
lineScene["line"] = LineString([[1, 2], [10, 10]]).buffer(.1)

print("intersects:", lineScene["disc"].intersects(lineScene["line"]))
print("distance:", lineScene["disc"].distance(lineScene["line"]))

ax = drawScene(lineScene, figsize=(4, 4))

Definition of a CollisionChecker
================================

The collision checker hides the obstacle representation from the planner. The planner only needs to ask:

* Is this point/configuration in collision?
* Is the local connection between two configurations in collision?

A minimal version needs only a few lines:

In [ ]:
class SimpleCollisionChecker(object):
    """Minimal collision checker: the planner can only ask, it never sees the obstacles."""

    def __init__(self, scene, limits=LIMITS):
        self.scene = scene
        self.limits = limits

    def pointInCollision(self, pos):
        """Collision or outside the limits -> True, free -> False"""
        x, y = pos
        if not (self.limits[0][0] <= x <= self.limits[0][1] and self.limits[1][0] <= y <= self.limits[1][1]):
            return True
        return any(obstacle.intersects(Point(x, y)) for obstacle in self.scene.values())

    def lineInCollisionExact(self, startPos, endPos):
        """Exact check of the line segment from startPos to endPos"""
        if self.pointInCollision(startPos) or self.pointInCollision(endPos):
            return True
        line = LineString([startPos, endPos])
        return any(obstacle.intersects(line) for obstacle in self.scene.values())

simpleChecker = SimpleCollisionChecker(lineScene)
print("[10, 10] in collision:", simpleChecker.pointInCollision([10, 10]))
print("[2, 5] in collision:", simpleChecker.pointInCollision([2, 5]))

The collision checker used by all planners of this course is `CollisionChecker` in `IPEnvironment.py`. It has the same interface plus:

* `getDim()` and `getEnvironmentLimits()`, so planners can sample configurations,
* `lineInCollision()`, a **sampled** check of the line (see below),
* `drawObstacles(ax)` for visualization,
* every check is recorded by `IPPerfMonitor` (number of calls and time).

From now on this class is used (show its source with `CollisionChecker??`).

In [ ]:
from IPEnvironment import CollisionChecker

cc = CollisionChecker(lineScene, limits=LIMITS)

Testing and using the collision checker
---------------------------------------

First single points: the first point lies on an obstacle, the second one is free, and the third one is outside the environment limits.

In [ ]:
testPoints = [[10, 10], [2, 5], [23, 5]]
for point in testPoints:
    print(point, "in collision (or outside limits):", cc.pointInCollision(point))

ax = drawScene(lineScene, limits=[[0, 24], [0, 22]])
for point in testPoints:
    if cc.pointInCollision(point):
        ax.scatter(*point, marker="x", s=100, linewidths=2.5, color=COLLISION_COLOR, zorder=5)
    else:
        ax.scatter(*point, marker="o", s=80, color=FREE_COLOR, zorder=5)
ax.axvline(LIMITS[0][1], color="0.5", linestyle="--", linewidth=1)   # border of the environment
ax.set_title("blue: free, orange: collision")
plt.show()

Now local connections. The exact check `lineInCollisionExact` tests the line segment as a whole. The sampled check `lineInCollision` tests `steps + 1` points on the line (default: 40 steps), like a planner which can only check configurations.

In [ ]:
testLines = [([5, 0], [5, 10]), ([5, 0], [10, 0])]

ax = drawScene(lineScene)
for start, end in testLines:
    collision = cc.lineInCollisionExact(start, end)
    print(start, "->", end, " sampled:", cc.lineInCollision(start, end), " exact:", collision)
    ax.plot([start[0], end[0]], [start[1], end[1]], linewidth=3,
            color=COLLISION_COLOR if collision else FREE_COLOR)

Sampled vs. exact line check
----------------------------

In the example above both checks give the same result. But the sampled check can **miss thin obstacles**: only the sampled points are checked, not the space in between. Example: a wall, which is only 0.1 wide.

In [ ]:
thinWall = {"wall": LineString([(10.2, 2), (10.2, 20)]).buffer(0.05)}
ccWall = CollisionChecker(thinWall, limits=LIMITS)
start, end = [1, 11], [21, 11]
steps = 40

print("sampled:", ccWall.lineInCollision(start, end, steps=steps))
print("exact:  ", ccWall.lineInCollisionExact(start, end))

ax = drawScene(thinWall, figsize=(6, 6))
samples = [[start[0] + i / steps * (end[0] - start[0]), start[1] + i / steps * (end[1] - start[1])]
           for i in range(steps + 1)]
ax.plot([start[0], end[0]], [start[1], end[1]], color="0.6", linewidth=1)
ax.scatter([p[0] for p in samples], [p[1] for p in samples], s=12, color=FREE_COLOR, zorder=5)
ax.set_title(f"{steps} steps: distance of the samples {(end[0] - start[0]) / steps}")

# zoom into the region of the wall
ax.set_xlim(8, 12.5)
ax.set_ylim(9.5, 12.5)
plt.show()

The wall is detected only if a sample point lies inside it. How many steps are needed?

In [ ]:
for steps in [40, 80, 160, 320]:
    print(f"{steps:4d} steps (distance {20 / steps:.3f}): collision detected = {ccWall.lineInCollision(start, end, steps=steps)}")

Collision checks are the expensive part
---------------------------------------

More steps detect thinner obstacles, but every step is one more collision check. In path planning, the number of collision checks is usually the dominating cost. `IPPerfMonitor` records every call of the collision checker:

In [ ]:
from IPPerfMonitor import IPPerfMonitor

IPPerfMonitor.clearData()
ccWall.lineInCollision(start, end, steps=40)
ccWall.lineInCollision(start, end, steps=320)
ccWall.lineInCollisionExact(start, end)

stats = IPPerfMonitor.dataFrame()
stats.groupby("name")["time"].agg(["count", "sum"])

Every call of `lineInCollision` checks its sample points with `pointInCollision` (until the first collision is found), therefore `pointInCollision` is called many times. Later, the planners are compared by exactly these numbers (see IP-X-0 / IP-X-1).

Benchmark scenes
================

A benchmark combines a scene, a collision checker, one or more start configurations, one or more goal configurations, a short description, and a difficulty level. This makes it easier to compare planners on the same problems.

In [ ]:
from IPBenchmark import Benchmark
import IPTestSuite as ts

As an example the Christmas tree from above: the goal lies deep inside a narrow pocket between two branches.

In [ ]:
description = "The goal lies deep inside a narrow pocket between the branches."
treeBenchmark = Benchmark("Christmas tree", CollisionChecker(treeField, limits=LIMITS), [[2, 20]], [[9.5, 9.5]],
                          description, 4)

Validating and assessing a benchmark
------------------------------------

A benchmark is only useful if its start and goal configurations are valid. For a 2D point robot Shapely can even answer, whether a solution exists at all: the free space (environment minus obstacles) may consist of several separated parts, start and goal have to lie in the same part.

How difficult a benchmark is, can be estimated with the idea of IP-2-0: the obstacles are grown by a radius `r` (C-space of a disc robot). The largest `r`, for which start and goal are still connected, is the **clearance** of the benchmark: half the width of the narrowest passage, which has to be passed. The smaller the clearance, the harder it is for sampling-based planners to find the passage.

In [ ]:
def freeSpace(collisionChecker, radius=0.0):
    """Free space of a disc robot with the given radius (radius 0: point robot).
    Simplification: only the obstacles are grown, not the border of the environment."""
    limits = collisionChecker.getEnvironmentLimits()
    area = box(limits[0][0], limits[1][0], limits[0][1], limits[1][1])
    obstacles = unary_union(list(collisionChecker.scene.values()))
    return area.difference(obstacles.buffer(radius))

def isSolvable(benchmark, radius=0.0):
    """Is there a start and a goal in the same connected part of the free space?"""
    free = freeSpace(benchmark.collisionChecker, radius)
    for part in getattr(free, "geoms", [free]):
        if (any(part.covers(Point(start)) for start in benchmark.startList)
                and any(part.covers(Point(goal)) for goal in benchmark.goalList)):
            return True
    return False

def clearance(benchmark, maxRadius=5.0, tolerance=0.01):
    """Largest radius r (bisection), for which the benchmark is still solvable."""
    if not isSolvable(benchmark):
        return 0.0
    low, high = 0.0, maxRadius
    while high - low > tolerance:
        middle = (low + high) / 2
        if isSolvable(benchmark, middle):
            low = middle
        else:
            high = middle
    return low

def levelFromClearance(c):
    """Rule of thumb used for the levels in IPTestSuite.py"""
    return 1 if c >= 2.0 else 2 if c >= 0.75 else 3 if c >= 0.4 else 4

def validateBenchmark(benchmark):
    """Print validity, solvability and difficulty measures of a benchmark. Returns True if it is valid."""
    checker = benchmark.collisionChecker
    valid = True
    for kind, configurations in (("start", benchmark.startList), ("goal", benchmark.goalList)):
        for q in configurations:
            free = not checker.pointInCollision(q)
            print(f"  {kind} {q} valid: {free}")
            valid = valid and free
    limits = checker.getEnvironmentLimits()
    area = box(limits[0][0], limits[1][0], limits[0][1], limits[1][1])
    c = clearance(benchmark)
    print(f"  solvable: {isSolvable(benchmark)}")
    print(f"  free space: {freeSpace(checker).area / area.area:.0%} of the environment")
    print(f"  clearance: {c:.2f} -> level {levelFromClearance(c)} (given level: {benchmark.level})")
    return valid

In [ ]:
print(treeBenchmark.name)
validateBenchmark(treeBenchmark)

ax = drawScene(treeBenchmark.collisionChecker.scene, starts=treeBenchmark.startList, goals=treeBenchmark.goalList,
               title=treeBenchmark.name, figsize=(6, 6))

The clearance can be made visible (zoomed into the pocket with the goal; dark: obstacles, light: grown by r). With a radius slightly smaller than the clearance the pocket is still reachable, a slightly larger radius closes it: start and goal are separated.

In [ ]:
c = clearance(treeBenchmark)
obstacles = unary_union(list(treeBenchmark.collisionChecker.scene.values()))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, radius in zip(axes, [c - 0.05, c + 0.05]):
    drawScene({"grown": obstacles.buffer(radius)}, goals=treeBenchmark.goalList, ax=ax,
              title=f"r = {radius:.2f}: solvable = {isSolvable(treeBenchmark, radius)}")
    plotting.plot_polygon(obstacles, ax=ax, add_points=False, color=OBSTACLE_COLOR, facecolor=OBSTACLE_COLOR)
    ax.set_xlim(4, 12)
    ax.set_ylim(7, 12.5)
plt.show()

Existing test suite
-------------------

The repository contains a small benchmark suite in `IPTestSuite.py`. It is used by the planning notebooks. The loop below validates every benchmark and draws it.

In [ ]:
fig, axes = plt.subplots(1, len(ts.benchList), figsize=(4.5 * len(ts.benchList), 4.5))
for ax, benchmark in zip(axes, ts.benchList):
    print(f"--- {benchmark.name} (level {benchmark.level}) ---")
    print(benchmark.description)
    validateBenchmark(benchmark)
    drawScene(benchmark.collisionChecker.scene, starts=benchmark.startList, goals=benchmark.goalList,
              title=f"{benchmark.name} (level {benchmark.level})", ax=ax)
    ax.get_legend().remove()

Remarks:

* "Bottleneck" has the most free space (96 %), but the smallest clearance: the passage is hard to find for sampling-based planners although the environment looks almost empty.
* "Fat bottleneck" has the same clearance, but the passage is much longer. The clearance does not capture this, therefore the level is only a rough guide.
* "Trap" has a large clearance (level 1) and is easy for sampling-based planners. It is meant for planners, which head straight for the goal (e.g. greedy search strategies).

# Exercise

1. Design your own benchmark of level 3 and check it with `validateBenchmark`.
2. Find a line in `treeField`, which the sampled check `lineInCollision` (40 steps) misses. How many steps are needed to detect the collision?
3. Build a trap for a planner, which always moves straight towards the goal.
4. Create a benchmark without a solution. What should a planner report in this case?